# Sprint 1 — Baseline on English voice cloning clips

Run the real vs cloned English clips in `test_data/` through the model and `evaluate.py`.
Save the JSON to `results/sprint1_baseline_english.json` (do NOT overwrite any existing run).

In [ ]:
# @title 1. Mount Drive + clone repo + hydrate dataset (folium-style)
from google.colab import drive
from pathlib import Path
import sys, os, pathlib, subprocess

drive.mount("/content/drive")

REPO_URL = "https://github.com/io-PEAK/VoxDetect.git"   # change if forked
REPO_DIR = Path("/content/VoxDetect")                    # session-only git clone

# ---- Durable on Google Drive (persist across sessions) ----
ML_BASE        = Path("/content/drive/MyDrive/VoxDetect/ml-core")
DATASET_DIR    = ML_BASE / "dataset"        # test_data.zip + .manifest.json (upload once)
RESULTS_DIR    = ML_BASE / "results"        # every evaluate run writes a UNIQUE <sprint>.json
CHECKPOINT_DIR = ML_BASE / "checkpoints"     # model artifacts
results_dir = RESULTS_DIR
for d in (DATASET_DIR, RESULTS_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- Session-local (rebuilt each run from the Drive archive) ----
LOCAL_DATA_DIR = Path("/content/VoxDetect_data")   # hydrated test_data (real/ + cloned/)

# ---- git clone the repo (session-only) so we get latest src/ + scripts/ ----
if not (REPO_DIR / "ml-core" / "src").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

SRC_PKG = REPO_DIR / "ml-core" / "src"
sys.path.insert(0, str(SRC_PKG))

# ---- hydrate local test_data from the Drive archive (mirrors folium organize_datasets) ----
subprocess.run([
    sys.executable, str(REPO_DIR / "ml-core" / "scripts" / "organize_dataset.py"),
    "--raw-dir", str(DATASET_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
])
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
test_data_dir = LOCAL_DATA_DIR   # evaluate.py reads real/ + cloned/ from here

print("src/ package at:", SRC_PKG)
print("DATASET_DIR (Drive):", DATASET_DIR)
print("LOCAL_DATA_DIR   :", LOCAL_DATA_DIR)
print("RESULTS_DIR (Drive):", RESULTS_DIR)
print("CHECKPOINT_DIR  :", CHECKPOINT_DIR)

# Install audio + ML deps (no stray 'audio' package)
!pip install -q torch torchaudio librosa soundfile transformers resemblyzer huggingface_hub numpy scipy
print("deps installed")

In [ ]:
# @title Run baseline on English clips
from detect import DetectionEngine

# cell 1 already hydrated the dataset from Drive:
#   test_data_dir = /content/VoxDetect_data (local, real/ + cloned/)
#   results_dir   = /content/drive/MyDrive/VoxDetect/ml-core/results (DRIVE)

# Run evaluate over the LOCAL hydrated clips, saving a UNIQUE json to the DRIVE
# results/ (never overwrites an earlier run).
import subprocess, sys, os
out = str(results_dir / "sprint1_baseline_english.json")
result = subprocess.run([
    sys.executable, "-m", "evaluate",
    "--root", str(test_data_dir),
    "--variant", "wav2vec2",
    "--out", out,
    "--find-threshold",
], cwd=str(SRC_PKG))
assert result.returncode == 0, "evaluate failed"
print("\nSaved ->", out, "(on Drive; paste numbers into results.md)")